In [1]:
pip install ultralytics opencv-python numpy pandas torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 54.0 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import pandas as pd

def audit_png_dataset(base_path):
    stats = []
    splits = ['Train', 'Test']
    
    for split in splits:
        split_path = os.path.join(base_path, split)
        if not os.path.exists(split_path): continue
            
        classes = [d for d in os.listdir(split_path) if os.path.isdir(os.path.join(split_path, d))]
        
        for cls in classes:
            cls_path = os.path.join(split_path, cls)
            # Count PNG files
            images = [f for f in os.listdir(cls_path) if f.endswith('.png')]
            
            stats.append({
                'Split': split,
                'Class': cls,
                'Image Count': len(images)
            })
            
    return pd.DataFrame(stats)

# --- Execution ---
dataset_path = "/kaggle/input/datasets/odins0n/ucf-crime-dataset" 
print("Test Data")
df_png_stats = audit_png_dataset(dataset_path)
print(df_png_stats[df_png_stats['Split'] == 'Test'].sort_values(by='Image Count', ascending=False))

print("Train Data")
df_png_stats = audit_png_dataset(dataset_path)
print(df_png_stats[df_png_stats['Split'] == 'Train'].sort_values(by='Image Count', ascending=False))

Test Data
   Split          Class  Image Count
19  Test   NormalVideos        64952
25  Test       Burglary         7657
18  Test       Shooting         7630
22  Test    Shoplifting         7623
21  Test      Explosion         6510
17  Test         Arrest         3365
20  Test          Arson         2793
14  Test  RoadAccidents         2663
15  Test        Assault         2657
24  Test       Stealing         1984
27  Test       Fighting         1231
16  Test      Vandalism         1111
23  Test        Robbery          835
26  Test          Abuse          297
Train Data
    Split          Class  Image Count
5   Train   NormalVideos       947768
10  Train       Stealing        44802
9   Train        Robbery        41493
11  Train       Burglary        39504
3   Train         Arrest        26397
8   Train    Shoplifting        24835
13  Train       Fighting        24684
6   Train          Arson        24421
0   Train  RoadAccidents        23486
12  Train          Abuse        19076
7   Tr

In [3]:
# import os
# import glob
# import numpy as np
# from ultralytics import YOLO
# from tqdm import tqdm

# # Load model and move it to GPU
# model = YOLO('yolov8n.pt').to('cuda') 

# def extract_features_layer0_gpu(class_folder, save_path, seq_len=30, step=5, max_images=None):
#     image_paths = sorted(glob.glob(os.path.join(class_folder, "**/*.png"), recursive=True))
    
#     if max_images:
#         image_paths = image_paths[:max_images]

#     if len(image_paths) < seq_len:
#         print(f"Empty folder: {class_folder}")
#         return

#     all_sequences = []
    
#     # Pre-calculating indices for sliding windows
#     indices = list(range(0, len(image_paths) - seq_len, step))
    
#     for i in tqdm(indices, desc=f"GPU Extracting {os.path.basename(class_folder)}"):
#         window_paths = image_paths[i : i + seq_len]
        
#         # Batch inference: YOLOv8 is much faster when processing a list of paths
#         # We use a smaller batch size to avoid OOM (Out of Memory) errors
#         results = model.predict(window_paths, verbose=False, device=0)
        
#         window_features = []
#         for res in results:
#             classes = res.boxes.cls.cpu().numpy()
            
#             p_count = np.sum(classes == 0)
#             v_count = np.sum(np.isin(classes, [2, 3, 5, 7]))
#             m_conf = np.max(res.boxes.conf.cpu().numpy()) if len(res.boxes) > 0 else 0
            
#             window_features.append([p_count, v_count, m_conf])
            
#         all_sequences.append(window_features)
    
#     np.save(save_path, np.array(all_sequences))

# # --- EXECUTION ---
# BASE_PATH = "/kaggle/input/datasets/odins0n/ucf-crime-dataset/Train" 
# OUT_DIR = "/kaggle/working/features"
# os.makedirs(OUT_DIR, exist_ok=True)

# EXCLUDE = ['Arson', 'Explosion']

# for folder in os.listdir(BASE_PATH):
#     folder_path = os.path.join(BASE_PATH, folder)
#     if os.path.isdir(folder_path) and folder not in EXCLUDE:
#         save_file = os.path.join(OUT_DIR, f"{folder}_feat.npy")
        
#         if os.path.exists(save_file):
#             continue
            
#         # Keep the limit for NormalVideos to protect your GPU time!
#         limit = 50000 if folder == "NormalVideos" else None
#         extract_features_layer0_gpu(folder_path, save_file, step=5, max_images=limit)

In [4]:
import os
import glob
import numpy as np
from ultralytics import YOLO
from tqdm import tqdm

model = YOLO('yolov8n.pt').to('cuda') 

def extract_features_test_logic(class_folder, save_path, seq_len=30, step=5, max_images=None):
    image_paths = sorted(glob.glob(os.path.join(class_folder, "**/*.png"), recursive=True))
    
    if max_images:
        image_paths = image_paths[:max_images]

    if len(image_paths) < seq_len:
        return

    all_sequences = []
    
    for i in tqdm(range(0, len(image_paths) - seq_len, step), desc=f"Test Set: {os.path.basename(class_folder)}"):
        window_paths = image_paths[i : i + seq_len]
        results = model.predict(window_paths, verbose=False, device=0)
        
        window_features = []
        for res in results:
            classes = res.boxes.cls.cpu().numpy()
            p_count = np.sum(classes == 0)
            v_count = np.sum(np.isin(classes, [2, 3, 5, 7]))
            m_conf = np.max(res.boxes.conf.cpu().numpy()) if len(res.boxes) > 0 else 0
            window_features.append([p_count, v_count, m_conf])
            
        all_sequences.append(window_features)
    
    np.save(save_path, np.array(all_sequences))

# --- EXECUTION ---
TEST_BASE_PATH = "/kaggle/input/datasets/odins0n/ucf-crime-dataset/Test" 
TEST_OUT_DIR = "/kaggle/working/test_features"
os.makedirs(TEST_OUT_DIR, exist_ok=True)

EXCLUDE = ['Arson', 'Explosion']

for folder in os.listdir(TEST_BASE_PATH):
    folder_path = os.path.join(TEST_BASE_PATH, folder)
    if os.path.isdir(folder_path) and folder not in EXCLUDE:
        save_file = os.path.join(TEST_OUT_DIR, f"TEST_{folder}_feat.npy")
        
        # MAINTAINING RATIO: Using ~3,426 images for NormalVideos in Test
        limit = 3426 if folder == "NormalVideos" else None
        
        extract_features_test_logic(folder_path, save_file, step=5, max_images=limit)

print("\n--- RATIO-BALANCED TEST EXTRACTION COMPLETE ---")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


Test Set: Fighting: 100%|██████████| 241/241 [01:06<00:00,  3.64it/s]


--- RATIO-BALANCED TEST EXTRACTION COMPLETE ---
